In [1]:
# Install rapidfuzz
!pip install rapidfuzz


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import seaborn as sns
import matplotlib.pyplot as plt
from rapidfuzz import process, fuzz

import warnings
import pandas as pd
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=(SettingWithCopyWarning))

# Download Data

In [3]:
#From open states: bill_sponsorship
sponsorships=pd.read_csv('US_119_bill_sponsorships.csv')

#Congressional list with bioguide ids and districts
meta_data=pd.read_csv('119th Congress.csv')

#From open states: vote_counts (by vote_id; raw counts of individual votes)
counts=pd.read_csv('US_119_vote_counts.csv')

#From open states: votes (vote identifier; type of vote, result of vote,d ate of vote, bill_id)
votes=pd.read_csv('US_119_votes.csv')

#From open states: bills (bill information)
bills=pd.read_csv('US_119_bills.csv')

#From open states: vote_people (individual votes)
ind_votes=pd.read_csv('US_119_vote_people.csv')

In [4]:
meta_data.head()

,Name,Chamber,bioguide_id,State,District,Party
0,Alma S. Adams,House,A000370,North Carolina,12,Democratic
1,Robert B. Aderholt,House,A000055,Alabama,4,Republican
2,Pete Aguilar,House,A000371,California,33,Democratic
3,Mark Alford,House,A000379,Missouri,4,Republican
4,Rick W. Allen,House,A000372,Georgia,12,Republican


# Match Names

The OpenStates data is not consistent on using bioguide_ids so this code was written to do a match on a shortened version of each congress person's name

In [5]:
#make a name_clean column for matching purposes
meta_data['Name'] = meta_data['Name'].str.replace(r'\b(jr|sr)\b\.?', '', case=False, regex=True)
sponsorships['name_clean'] = sponsorships['name'].str.lower().str.replace('.', '', regex=False).str.replace(',', '', regex=False)
meta_data['name_clean']=meta_data['Name'].str.lower().str.strip().str.replace('.', '', regex=False).str.replace(',', '', regex=False)

# Split the 'name_clean' column into three columns: first, middle, and last names.
def split_name(name):
    parts = name.split()
    if len(parts) == 1:
        return parts[0], '', ''
    elif len(parts) == 2:
        return parts[0], '', parts[1]
    elif len(parts) == 3:
        return parts[0], parts[1], parts[2]
    else:
        return parts[0], ' '.join(parts[1:-1]), parts[-1]


sponsorships[['first_name', 'middle_name', 'last_name']] = sponsorships['name_clean'].apply(lambda x: pd.Series(split_name(x)))
meta_data[['first_name', 'middle_name', 'last_name']] = meta_data['name_clean'].apply(lambda x: pd.Series(split_name(x)))

#limit first name to first 3 letters only
meta_data['first_name'] = meta_data['first_name'].str[:3]
sponsorships['first_name'] = sponsorships['first_name'].str[:3]

# make column that is short_name combining first_name and last_name columns
meta_data['short_name'] =  meta_data['last_name']+ ' ' + meta_data['first_name']
sponsorships['short_name'] = sponsorships['last_name']+ ' ' + sponsorships['first_name']


In [6]:
#function to perform fuzzy match on short_name columns
def fuzzy_merge_fast(df_1, df_2, key1, key2, threshold=70):
    choices = df_2[key2].dropna().unique().tolist()
    matches = df_1[key1].apply(
        lambda x: process.extractOne(x, choices, scorer=fuzz.token_sort_ratio, score_cutoff=threshold)
    )
    df_1['matched_name'] = matches.apply(lambda x: x[0] if x else None)
    return df_1

#run fuzzy_match on short_name columns
sponsorships = fuzzy_merge_fast(sponsorships, meta_data, 'short_name', 'short_name', threshold=70)

sponsorships.head(2)

,id,name,entity_type,organization_id,person_id,bill_id,primary,classification,name_clean,first_name,middle_name,last_name,short_name,matched_name
0,fd0c1569-ae08-49cd-af50-adaf0ef752bb,Chuck Grassley,person,NaN,ocd-person/259d896b-e9b5-5237-9243-f5597ae0db7c,ocd-bill/e03669bb-3352-4355-947e-da238094c93b,False,cosponsor,chuck grassley,chu,,grassley,grassley chu,grassley chu
1,21e79bb9-0fd0-43db-ab96-49940161af2c,Katie Britt,person,NaN,ocd-person/7bf077d8-e8b9-503e-b9c2-d18b55951e93,ocd-bill/98ee8b23-4866-4391-b212-392bbdd881c6,False,cosponsor,katie britt,kat,,britt,britt kat,britt kat


In [7]:
#find rows where there was a name match
matched = sponsorships[sponsorships['matched_name'].notna()].copy()

matched = matched.merge(
    meta_data[['short_name', 'bioguide_id', 'Party', 'Chamber', 'State', 'District']],
    left_on='matched_name',
    right_on='short_name',
    how='left'
)

matched.head(2)


,id,name,entity_type,organization_id,person_id,bill_id,primary,classification,name_clean,first_name,middle_name,last_name,short_name_x,matched_name,short_name_y,bioguide_id,Party,Chamber,State,District
0,fd0c1569-ae08-49cd-af50-adaf0ef752bb,Chuck Grassley,person,NaN,ocd-person/259d896b-e9b5-5237-9243-f5597ae0db7c,ocd-bill/e03669bb-3352-4355-947e-da238094c93b,False,cosponsor,chuck grassley,chu,,grassley,grassley chu,grassley chu,grassley chu,G000386,Republican,Senate,Iowa,NaN
1,21e79bb9-0fd0-43db-ab96-49940161af2c,Katie Britt,person,NaN,ocd-person/7bf077d8-e8b9-503e-b9c2-d18b55951e93,ocd-bill/98ee8b23-4866-4391-b212-392bbdd881c6,False,cosponsor,katie britt,kat,,britt,britt kat,britt kat,britt kat,B001319,Republican,Senate,Alabama,NaN


In [8]:
#find unmatched rows
unmatched = sponsorships[sponsorships['matched_name'].isna()].copy()
for col in ['bioguide_id', 'party', 'type', 'state', 'district']:
    unmatched[col] = None

In [9]:
#show unique names in unmatched dataframe
#(the results here should be delegates to congress which are not members and Marco Rubio -who left senate to be in cabinet and JD vance who is president of senate)
unmatched['name'].unique()


array(['James C. Moylan', 'Aumua Amata Coleman Radewagen',
       'Kimberlyn King-Hinds', 'Marco Rubio', 'J. D. Vance', 'Randy Fine',
       'Jimmy Patronis'], dtype=object)

In [10]:
# delete unnecessary columns from matched

matched = matched.drop(columns=['name_clean', 'first_name', 'middle_name', 'last_name', 'short_name_x', 'matched_name', 'short_name_y'])


# Create bill_sponsor dataframe

In [11]:
#Create dataframe of only primary sponsors of bills
primary_df = matched[matched['classification'] == 'primary'][['id','name','bill_id', 'bioguide_id', 'Party', 'Chamber']]
primary_df = primary_df.rename(columns={'Party': 'primary_party'})

#create dataframe of only cosponsors of bills
cosponsor_df = matched[matched['classification'] == 'cosponsor'][['name','bill_id', 'bioguide_id', 'Party', 'Chamber']]
cosponsor_df = cosponsor_df.rename(columns={'Party': 'cosponsor_party'})

In [12]:
#merge the primary and cosponsor dataframe
bill_sponsor = primary_df.merge(cosponsor_df, on='bill_id', how='left', suffixes=('_primary','_cosponsor'))

#add column to bill sponsor that shows True if bill has cross-party co-sponsor
bill_sponsor['Cross Party Sponsorship']=bill_sponsor['primary_party']!=bill_sponsor['cosponsor_party']




In [13]:
# Replace results in Cross Party Sponsorship to be true if any co-sponsor is cross party

# Group by 'bill_id' and check if 'Cross Party Sponsorship' is True for any row within the group
cross_party_sponsorship_by_bill = bill_sponsor.groupby('bill_id')['Cross Party Sponsorship'].any()

# Update the 'Cross Party Sponsorship' column in the original DataFrame based on the grouped results
bill_sponsor['Cross Party Sponsorship'] = bill_sponsor['bill_id'].map(cross_party_sponsorship_by_bill)


In [14]:
bill_sponsor.head(2)

,id,name_primary,bill_id,bioguide_id_primary,primary_party,Chamber_primary,name_cosponsor,bioguide_id_cosponsor,cosponsor_party,Chamber_cosponsor,Cross Party Sponsorship
0,0d502486-51a3-421f-b963-2e5c640c4df5,Jeanne Shaheen,ocd-bill/e03669bb-3352-4355-947e-da238094c93b,S001181,Democratic,Senate,Chuck Grassley,G000386,Republican,Senate,True
1,0a84a66d-bbce-4fde-9c15-697e00966191,Mike Lee,ocd-bill/c41c05d8-a598-4204-9e8f-e63de4b14874,L000577,Republican,Senate,Ted Budd,B001305,Republican,Senate,False


In [15]:
# Keep only unique bill_ids in merged_df and necessary columns
bill_sponsor = bill_sponsor.drop_duplicates(subset=['bill_id'])
bill_sponsor=bill_sponsor[['id','name_primary', 'bill_id', 'bioguide_id_primary', 'primary_party',
       'Chamber_primary', 'Cross Party Sponsorship']]

In [16]:
bill_sponsor.head(2)

,id,name_primary,bill_id,bioguide_id_primary,primary_party,Chamber_primary,Cross Party Sponsorship
0,0d502486-51a3-421f-b963-2e5c640c4df5,Jeanne Shaheen,ocd-bill/e03669bb-3352-4355-947e-da238094c93b,S001181,Democratic,Senate,True
1,0a84a66d-bbce-4fde-9c15-697e00966191,Mike Lee,ocd-bill/c41c05d8-a598-4204-9e8f-e63de4b14874,L000577,Republican,Senate,False


# Create Bill Vote Dataraframe

In [17]:
counts = counts.drop(columns=['id'])

In [18]:
# Add new columns to the votes DataFrame
votes['abstain'] = 0
votes['not voting'] = 0
votes['no'] = 0
votes['yes'] = 0

In [19]:
# Iterate through votes dataframe to make the vote results into columns instead of rows
for index, row in votes.iterrows():
    matching_counts = counts[counts['vote_event_id'] == row['id']]
    for _, count_row in matching_counts.iterrows():
        if 'abstain' in count_row['option']:
            votes.loc[index, 'abstain'] = count_row['value']
        if 'not voting' in count_row['option']:
            votes.loc[index, 'not voting'] = count_row['value']
        if 'no' in count_row['option']:
            votes.loc[index, 'no'] = count_row['value']
        if 'yes' in count_row['option']:
            votes.loc[index, 'yes'] = count_row['value']


In [20]:
# add total_votes column
votes['total_votes'] = votes['abstain'] + votes['not voting'] + votes['no'] + votes['yes']

In [21]:
votes.head(3)

,id,identifier,motion_text,motion_classification,start_date,result,organization_id,bill_id,bill_action_id,jurisdiction,session_identifier,abstain,not voting,no,yes,total_votes
0,ocd-vote/a31b4e67-7b3d-45b0-9bc7-3461a975d202,us-2025-lower-3,On Ordering the Previous Question,['passage'],2025-01-03T22:54:00+00:00,pass,ocd-organization/24af4233-d9b5-5933-91b2-51d29...,ocd-bill/2c089ed7-8815-4d47-b432-e93c7c9eb1df,NaN,United States,119,0,8,210,216,434
1,ocd-vote/f28036c3-7abf-4509-9b1f-b133b3e2e329,us-2025-lower-4,On Motion to Commit with Instructions,['passage'],2025-01-03T23:01:00+00:00,fail,ocd-organization/24af4233-d9b5-5933-91b2-51d29...,ocd-bill/2c089ed7-8815-4d47-b432-e93c7c9eb1df,NaN,United States,119,0,11,214,209,434
2,ocd-vote/86b3245c-b8fd-41dc-a080-e2c1a7f3a1a8,us-2025-lower-5,On Agreeing to the Resolution,['passage'],2025-01-03T23:08:00+00:00,pass,ocd-organization/24af4233-d9b5-5933-91b2-51d29...,ocd-bill/2c089ed7-8815-4d47-b432-e93c7c9eb1df,NaN,United States,119,0,10,209,215,434


In [22]:
# Merge votes and bills DataFrames
merged_votes_bills = votes.merge(bills, left_on='bill_id', right_on='id', how='left')

merged_votes_bills=merged_votes_bills[merged_votes_bills['classification']=="['bill']"]

In [23]:
# Keep only votes that are on passage in either house or senate

passage_votes = merged_votes_bills[merged_votes_bills['motion_text'].str.startswith('On Passage', na=False)]


In [24]:
sen_passage_votes=passage_votes[passage_votes['organization_classification']=='upper']
sen_passage_votes

,id_x,identifier_x,motion_text,motion_classification,start_date,result,organization_id,bill_id,bill_action_id,jurisdiction_x,...,yes,total_votes,id_y,identifier_y,title,classification,subject,session_identifier_y,jurisdiction_y,organization_classification
18,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,us-2025-upper-7,On Passage of the Bill S. 5,['passage'],2025-01-20T11:12:00+00:00,pass,ocd-organization/072da2ce-df81-52c3-9cc8-323e2...,ocd-bill/64132b9b-fdb2-4e7d-b1f5-41cd1d38cdc0,NaN,United States,...,64,99,ocd-bill/64132b9b-fdb2-4e7d-b1f5-41cd1d38cdc0,S 5,Laken Riley Act,['bill'],[],119,United States,upper
25,ocd-vote/0e184688-d1a0-42ef-8aa5-4fdfcd3a2211,us-2025-lower-23,On Passage,['passage'],2025-01-22T22:01:00+00:00,pass,ocd-organization/24af4233-d9b5-5933-91b2-51d29...,ocd-bill/64132b9b-fdb2-4e7d-b1f5-41cd1d38cdc0,NaN,United States,...,263,433,ocd-bill/64132b9b-fdb2-4e7d-b1f5-41cd1d38cdc0,S 5,Laken Riley Act,['bill'],[],119,United States,upper
82,ocd-vote/d50dfbc7-800e-4d4c-a4b6-b1df6a6746c1,us-2025-upper-127,On Passage of the Bill S. 331,['passage'],2025-03-14T06:21:00+00:00,pass,ocd-organization/072da2ce-df81-52c3-9cc8-323e2...,ocd-bill/ca6e8c6e-7713-4d50-b393-b8eb7b2f0597,NaN,United States,...,84,100,ocd-bill/ca6e8c6e-7713-4d50-b393-b8eb7b2f0597,S 331,HALT Fentanyl Act,['bill'],[],119,United States,upper


In [25]:
passage_votes=passage_votes[['id_x', 'motion_text', 'motion_classification',
       'result', 'bill_id', 'abstain', 'not voting', 'no',
       'yes', 'total_votes', 'identifier_y', 'title', 'classification',
       'organization_classification']]

# Rename columns
passage_votes = passage_votes.rename(columns={'id_x': 'vote_id'})
passage_votes = passage_votes.rename(columns={'identifier_y': 'Bill Number'})

In [26]:
# prompt: merge passage_votes and bill_sponsor dataframes on bill_id

# Merge passage_votes and bill_sponsor DataFrames
passage_votes = passage_votes.merge(bill_sponsor, on='bill_id', how='left')

passage_votes.head(2)


,vote_id,motion_text,motion_classification,result,bill_id,abstain,not voting,no,yes,total_votes,Bill Number,title,classification,organization_classification,id,name_primary,bioguide_id_primary,primary_party,Chamber_primary,Cross Party Sponsorship
0,ocd-vote/9ca8ab8f-f167-439b-9ce9-58c940210c6d,On Passage,['passage'],pass,ocd-bill/bf8ba42d-25e2-4f17-8a43-efd0018a4138,0,11,159,264,434,HR 29,Laken Riley Act,['bill'],lower,ca35399c-5135-4002-aa4b-3d3707e8f384,Mike Collins,C001129,Republican,House,True
1,ocd-vote/d060f8e7-ca85-463a-ac8f-1a4daeed2e54,On Passage,['passage'],pass,ocd-bill/4f5998f7-48fa-4d3e-8419-cd024bbe9a35,1,50,140,243,434,HR 23,Illegitimate Court Counteraction Act,['bill'],lower,0fc2c39c-34a0-445e-9151-ded7e691d24b,Chip Roy,R000614,Republican,House,False


In [27]:
# prompt: sum the total_votes column in passage_votes

total_votes_sum = passage_votes['total_votes'].sum()
print(total_votes_sum)


7659


In [28]:
# make list of vote_ids of passage votes
passage_votes_ids=passage_votes['vote_id'].unique()

In [29]:
#  change column name in ind_votes from note to voter_bioguide_id
ind_votes = ind_votes.rename(columns={'note': 'voter_bioguide_id'})


In [30]:
# Filter ind_votes to keep only rows where 'vote_event_id' is in passage_votes_ids
ind_votes = ind_votes[ind_votes['vote_event_id'].isin(passage_votes_ids)]

In [31]:
#this should be the same as the total_votes_sum above
len(ind_votes)

7659

In [32]:
ind_votes.head(3)

,id,vote_event_id,option,voter_name,voter_id,voter_bioguide_id
2564,6f724c57-62ad-4f51-aaa1-095e873663db,ocd-vote/b7c29079-1be4-4fd7-8f60-590682ca391b,no,Adams,ocd-person/76bfaf2b-8259-5f56-bce9-d1c8cd6c780d,A000370
3431,944d20ac-94b5-4caa-8b94-eb0763a39f3c,ocd-vote/f66a86ff-3923-44ac-bb38-9ebff7500b22,no,Adams,ocd-person/76bfaf2b-8259-5f56-bce9-d1c8cd6c780d,A000370
3432,2a06b5b3-c643-4c74-a060-f6dbfe610418,ocd-vote/f66a86ff-3923-44ac-bb38-9ebff7500b22,yes,Aderholt,ocd-person/1a69c825-1015-5d72-a8d8-a720d586c3b4,A000055


In [33]:
#create dataframe of house only votes (house members have
house_ind_votes=ind_votes[ind_votes['voter_bioguide_id'].str.len() >= 5]

In [34]:
# Filter ind_votes to keep rows where the length of the voter_bioguide_id column is less than 5.
sen_ind_votes=ind_votes[ind_votes['voter_bioguide_id'].str.len() < 5]

# Split the 'voter_name' column into two columns: 'Last Name' and 'State'.
# Escape the opening parenthesis in the split pattern to treat it as a literal character.
sen_ind_votes[['Last Name', 'State']] = sen_ind_votes['voter_name'].str.split(r"(", n=1, expand=True)

sen_ind_votes['Last Name']=sen_ind_votes['Last Name'].str.lower().str.strip()
sen_ind_votes['State'] = sen_ind_votes['State'].str[-3:-1]
sen_ind_votes.head(3)



,id,vote_event_id,option,voter_name,voter_id,voter_bioguide_id,Last Name,State
15202,e7d684c4-bcbc-4827-bf65-19ff98296dbe,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,no,Alsobrooks (D-MD),NaN,S428,alsobrooks,MD
15203,880e08ff-de48-499f-933c-b5141e2e5974,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,no,Baldwin (D-WI),NaN,S354,baldwin,WI
15204,92888a5b-34a5-4c84-a6f4-60fee8701299,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,yes,Banks (R-IN),NaN,S429,banks,IN


In [35]:
# prompt: in sen_ind_votes['Last Name'] for any string that has more than one word delete the first word

# Iterate through the 'Last Name' column and modify strings with more than one word
for index, row in sen_ind_votes.iterrows():
    last_name = row['Last Name']
    if ' ' in last_name:
        new_last_name = last_name.split(' ', 1)[1]  # Split at the first space and take the second part
        sen_ind_votes.loc[index, 'Last Name'] = new_last_name


In [36]:
senate_meta_data=meta_data[meta_data['Chamber']=='Senate']
senate_meta_data.head(3)

,Name,Chamber,bioguide_id,State,District,Party,name_clean,first_name,middle_name,last_name,short_name
435,Angela D. Alsobrooks,Senate,A000382,Maryland,NaN,Democratic,angela d alsobrooks,ang,d,alsobrooks,alsobrooks ang
436,Tammy Baldwin,Senate,B001230,Wisconsin,NaN,Democratic,tammy baldwin,tam,,baldwin,baldwin tam
437,Jim Banks,Senate,B001299,Indiana,NaN,Republican,jim banks,jim,,banks,banks jim


In [37]:
# Look up senators by last name to get correct bioguide_ids in ind_votes file

# Create an empty list to store the results
bioguide_ids = []

# Iterate through the 'voter_id' column in sen_ind_votes
for last_name in sen_ind_votes['Last Name']:
    # Look up the last name in the senate_meta_data DataFrame
    match = senate_meta_data[senate_meta_data['last_name'] == last_name]

    # If a match is found, append the corresponding bioguide_id to the list
    if not match.empty:
        bioguide_ids.append(match['bioguide_id'].iloc[0])
    else:
        bioguide_ids.append(None)  # Append None if no match is found

# Add the bioguide_ids as a new column to sen_ind_votes
sen_ind_votes['bioguide_id'] = bioguide_ids


In [38]:
sen_ind_votes.head(5)

,id,vote_event_id,option,voter_name,voter_id,voter_bioguide_id,Last Name,State,bioguide_id
15202,e7d684c4-bcbc-4827-bf65-19ff98296dbe,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,no,Alsobrooks (D-MD),NaN,S428,alsobrooks,MD,A000382
15203,880e08ff-de48-499f-933c-b5141e2e5974,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,no,Baldwin (D-WI),NaN,S354,baldwin,WI,B001230
15204,92888a5b-34a5-4c84-a6f4-60fee8701299,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,yes,Banks (R-IN),NaN,S429,banks,IN,B001299
15205,f862b085-ab6c-4d28-966d-8afac3a04bc5,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,yes,Barrasso (R-WY),NaN,S317,barrasso,WY,B001261
15206,10d184f0-1ddb-4b68-bf50-3824388265d8,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,no,Bennet (D-CO),NaN,S330,bennet,CO,B001267


In [39]:
# Delete the 'voter_bioguide_id' column from sen_ind_votes
sen_ind_votes = sen_ind_votes.drop(columns=['voter_bioguide_id', 'Last Name', 'State'])

# Rename the 'bioguide_id' column to 'voter_bioguide_id' in the 'meta_data' DataFrame
sen_ind_votes = sen_ind_votes.rename(columns={'bioguide_id': 'voter_bioguide_id'})



In [40]:
sen_ind_votes.head()

,id,vote_event_id,option,voter_name,voter_id,voter_bioguide_id
15202,e7d684c4-bcbc-4827-bf65-19ff98296dbe,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,no,Alsobrooks (D-MD),NaN,A000382
15203,880e08ff-de48-499f-933c-b5141e2e5974,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,no,Baldwin (D-WI),NaN,B001230
15204,92888a5b-34a5-4c84-a6f4-60fee8701299,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,yes,Banks (R-IN),NaN,B001299
15205,f862b085-ab6c-4d28-966d-8afac3a04bc5,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,yes,Barrasso (R-WY),NaN,B001261
15206,10d184f0-1ddb-4b68-bf50-3824388265d8,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,no,Bennet (D-CO),NaN,B001267


In [41]:
house_ind_votes.head(2)

,id,vote_event_id,option,voter_name,voter_id,voter_bioguide_id
2564,6f724c57-62ad-4f51-aaa1-095e873663db,ocd-vote/b7c29079-1be4-4fd7-8f60-590682ca391b,no,Adams,ocd-person/76bfaf2b-8259-5f56-bce9-d1c8cd6c780d,A000370
3431,944d20ac-94b5-4caa-8b94-eb0763a39f3c,ocd-vote/f66a86ff-3923-44ac-bb38-9ebff7500b22,no,Adams,ocd-person/76bfaf2b-8259-5f56-bce9-d1c8cd6c780d,A000370


In [42]:
sen_ind_votes_blank=sen_ind_votes[sen_ind_votes['voter_bioguide_id'].isna()]

#run blanks - should only be Marco Rubio
sen_ind_votes_blank

,id,vote_event_id,option,voter_name,voter_id,voter_bioguide_id
15276,cdcfd815-c445-43d5-b1a7-a0d170e67c9c,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,yes,Rubio (R-FL),NaN,None


In [43]:
#recombine house and senate data
ind_votes=pd.concat([sen_ind_votes,house_ind_votes])

In [44]:
# Merge ind_votes with meta_data DataFrame
ind_votes = ind_votes.merge(meta_data, left_on='voter_bioguide_id', right_on='bioguide_id', how='left')


In [45]:
ind_votes.head(2)

,id,vote_event_id,option,voter_name,voter_id,voter_bioguide_id,Name,Chamber,bioguide_id,State,District,Party,name_clean,first_name,middle_name,last_name,short_name
0,e7d684c4-bcbc-4827-bf65-19ff98296dbe,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,no,Alsobrooks (D-MD),NaN,A000382,Angela D. Alsobrooks,Senate,A000382,Maryland,NaN,Democratic,angela d alsobrooks,ang,d,alsobrooks,alsobrooks ang
1,880e08ff-de48-499f-933c-b5141e2e5974,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,no,Baldwin (D-WI),NaN,B001230,Tammy Baldwin,Senate,B001230,Wisconsin,NaN,Democratic,tammy baldwin,tam,,baldwin,baldwin tam


In [46]:
#drop voter_name, voter_bioguide_id, state, district columns from ind_votes df
ind_votes = ind_votes.drop(columns=['voter_name', 'voter_bioguide_id', 'State', 'District', 'name_clean','first_name','middle_name','last_name','short_name'])


In [47]:
ind_votes.head(2)

,id,vote_event_id,option,voter_id,Name,Chamber,bioguide_id,Party
0,e7d684c4-bcbc-4827-bf65-19ff98296dbe,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,no,NaN,Angela D. Alsobrooks,Senate,A000382,Democratic
1,880e08ff-de48-499f-933c-b5141e2e5974,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,no,NaN,Tammy Baldwin,Senate,B001230,Democratic


In [48]:
# iterate over ind_votes df, matching on ind_votes['vote_event_id'] and passage_votes['vote_id'], adding a column to ind_votes with passage_votes['primary_party']

# Add a new column to ind_votes to store the primary party
ind_votes['primary_party'] = None

# Iterate over the ind_votes DataFrame
for index, row in ind_votes.iterrows():
  # Find the matching row in passage_votes based on vote_event_id
  matching_rows = passage_votes[passage_votes['vote_id'] == row['vote_event_id']]
  # If a match is found, update the primary_party column in ind_votes
  if not matching_rows.empty:
    ind_votes.loc[index, 'primary_party'] = matching_rows['primary_party'].iloc[0]
    ind_votes.loc[index, 'Cross Party Sponsorship'] = matching_rows['Cross Party Sponsorship'].iloc[0]


In [49]:
# Rename columns to clarify parties
ind_votes = ind_votes.rename(columns={'primary_party': 'Bill_Sponsor_Party'})
ind_votes = ind_votes.rename(columns={'Party': 'Voter_Party'})

In [50]:
ind_votes.head(2)

,id,vote_event_id,option,voter_id,Name,Chamber,bioguide_id,Voter_Party,Bill_Sponsor_Party,Cross Party Sponsorship
0,e7d684c4-bcbc-4827-bf65-19ff98296dbe,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,no,NaN,Angela D. Alsobrooks,Senate,A000382,Democratic,Republican,True
1,880e08ff-de48-499f-933c-b5141e2e5974,ocd-vote/9dffa967-bec7-4bd0-9522-32a79e9286e3,no,NaN,Tammy Baldwin,Senate,B001230,Democratic,Republican,True


In [51]:
#add boolean columns to determine if specific vote meets critera

ind_votes['Vote for Other Party']=((ind_votes['option']=='yes')&(ind_votes['Bill_Sponsor_Party']!=ind_votes['Voter_Party']))
ind_votes['Vote Against Own Party']=((ind_votes['Voter_Party']==ind_votes['Bill_Sponsor_Party'])&(ind_votes['option']=='no'))
ind_votes['Vote for Cosponsored Bill']=((ind_votes['option']=='yes')&(ind_votes['Cross Party Sponsorship']==True))

# Calculate the sum of those categories
sum_vote_for_other_party = ind_votes['Vote for Other Party'].sum()
sum_vote_against_party = ind_votes['Vote Against Own Party'].sum()
sum_for_cosponsored = ind_votes['Vote for Cosponsored Bill'].sum()

#Print sums
print('Votes for other party:',sum_vote_for_other_party)
print('Votes against own party:',sum_vote_against_party )
print('Vote for Cosponsored Bill',sum_for_cosponsored)

Votes for other party: 812
Votes against own party: 7
Vote for Cosponsored Bill 2201


In [52]:
#Keep only the rows that have a True in one of the relevant columns
cross_votes=ind_votes[(ind_votes['Vote for Other Party'] == True) |
                           (ind_votes['Vote Against Own Party'] == True) |
                           (ind_votes['Vote for Cosponsored Bill'] == True)]

In [53]:
#create a summary dataframe with only name, bioguide_id and vote counts

cross_votes_summary = cross_votes.groupby('Name').agg(
    {'Vote for Other Party': 'sum', 'Vote Against Own Party': 'sum', 'Vote for Cosponsored Bill':'sum'}
).reset_index()

# Merge with meta_data dataframe to get bioguide_id
cross_votes_summary = cross_votes_summary.merge(meta_data[['Name', 'bioguide_id']], on='Name', how='left')

# Reorder columns
cross_votes_summary = cross_votes_summary[['Name', 'bioguide_id', 'Vote for Other Party', 'Vote Against Own Party','Vote for Cosponsored Bill']]

cross_votes_summary.head()


,Name,bioguide_id,Vote for Other Party,Vote Against Own Party,Vote for Cosponsored Bill
0,Aaron Bean,B001314,0,0,7
1,Abraham Hamadeh,H001098,0,0,7
2,Adam Gray,G000605,11,0,6
3,Adam Smith,S000510,1,0,1
4,Addison McDowell,M001240,0,0,7


In [54]:
# Merge cross_votes_summary with ind_votes on 'bioguide_id'
meta_data = meta_data.merge(cross_votes_summary, on='bioguide_id', how='left')

In [55]:
# prompt: drop name_y column from meta_data df and change Name_X column to Name

meta_data = meta_data.drop(columns=['Name_y','name_clean','first_name','middle_name','last_name','short_name'])
meta_data = meta_data.rename(columns={'Name_x': 'Name'})


In [56]:
meta_data.head()

,Name,Chamber,bioguide_id,State,District,Party,Vote for Other Party,Vote Against Own Party,Vote for Cosponsored Bill
0,Alma S. Adams,House,A000370,North Carolina,12,Democratic,1.0,0.0,1.0
1,Robert B. Aderholt,House,A000055,Alabama,4,Republican,0.0,0.0,7.0
2,Pete Aguilar,House,A000371,California,33,Democratic,2.0,0.0,2.0
3,Mark Alford,House,A000379,Missouri,4,Republican,0.0,0.0,7.0
4,Rick W. Allen,House,A000372,Georgia,12,Republican,0.0,0.0,7.0


In [57]:
# Fill NaN values in 'Vote for Other Party' and 'Vote Against Own Party' with 0
meta_data.fillna({'Vote for Other Party': 0}, inplace=True)
meta_data.fillna({'Vote Against Own Party': 0}, inplace=True)
meta_data.fillna({'Vote for Cosponsored Bill': 0}, inplace=True)

# Display the updated ind_votes DataFrame
meta_data.head(2)

,Name,Chamber,bioguide_id,State,District,Party,Vote for Other Party,Vote Against Own Party,Vote for Cosponsored Bill
0,Alma S. Adams,House,A000370,North Carolina,12,Democratic,1.0,0.0,1.0
1,Robert B. Aderholt,House,A000055,Alabama,4,Republican,0.0,0.0,7.0


In [59]:
meta_data.to_csv('meta_data_with_votes.csv', index=False)